# Notebook 25: Production-Ready ML Pipelines with scikit-learn
### Part 25/30 – ML Mastery Series for Python Experts

## Why Raw Notebooks Fail in Production – Pipeline Necessity

You built beautiful models — now let's make them bulletproof, reproducible, and ready to ship to production. Pipelines are the bridge from notebook hero to engineering reality…

- **Data leakage via fit on full data**: Fitting scalers/imputers on entire dataset before train/test split leaks test information into training
- **Inconsistent train/test transforms**: Manual preprocessing steps often differ between training and inference, causing subtle bugs
- **Code duplication**: Copy-pasting preprocessing code across experiments creates maintenance nightmares and version drift
- **Hard to reproduce**: Ad-hoc cell execution order and hidden state make notebooks impossible to rerun reliably
- **Deployment mismatch**: Development code rarely matches production serving code, causing "works on my machine" failures
- **No caching → slow CV**: Recomputing expensive preprocessing (feature engineering, text embeddings) on every CV fold wastes hours
- **Difficult versioning**: Without unified pipelines, tracking which preprocessing matched which model is error-prone
- **No serialization of full workflow**: Saving just the model loses preprocessing steps; pipelines save the complete workflow

## Learning Objectives

By the end of this notebook, you will be able to:

- **Build full Pipeline with preprocessing + model**: Chain transformers and estimators into a single reproducible workflow
- **Use ColumnTransformer for mixed types**: Handle numeric and categorical features with different preprocessing in one unified pipeline
- **Create custom transformers safely**: Implement `BaseEstimator` and `TransformerMixin` for domain-specific feature engineering
- **Combine features with FeatureUnion**: Merge multiple feature extraction strategies (raw, PCA, polynomial) into one pipeline
- **Cache expensive steps**: Use `memory` parameter to speed up repeated cross-validation without code changes
- **Avoid leakage with make_pipeline & cross_val_predict**: Ensure proper fit/transform boundaries and safe nested cross-validation
- **Serialize & load pipelines with joblib**: Save complete workflows (preprocessing + model) for production deployment
- **Debug pipeline steps & named access**: Inspect intermediate steps, extract fitted attributes, and troubleshoot effectively
- **Version & reproduce entire workflow**: Track complete preprocessing and modeling configurations for experiment management
- **Prepare for deployment (predict only)**: Load saved pipelines and generate predictions on new data without retraining

## 🛠️ 1. Basic Pipeline – From Raw to Clean

The Pipeline class chains multiple estimators into a single unified object. When you call `fit()`, each transformer fits on the output of the previous step, and the final estimator fits on the transformed data. When you call `predict()`, the same transformations are applied in sequence.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

# Core sklearn imports
from sklearn.datasets import load_breast_cancer, fetch_california_housing, load_iris
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectFromModel, RFE
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, make_pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, mean_squared_error
import sklearn
from joblib import Memory

# Set seeds and style
np.random.seed(42)
sns.set_theme(style="whitegrid")
%matplotlib inline

print("Libraries loaded successfully")
print(f"Scikit-learn version: {sklearn.__version__ if 'sklearn' in dir() else '1.3.x'}")

Libraries loaded successfully
Scikit-learn version: 1.6.1


In [2]:
# Load breast cancer dataset for classification example
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

print(f"Dataset: Breast Cancer Wisconsin")
print(f"Features: {X.shape[1]} ({X.shape[0]} samples)")
print(f"Target: Binary classification (malignant/benign)")
print(f"Feature names: {feature_names[:3]}...")

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")

Dataset: Breast Cancer Wisconsin
Features: 30 (569 samples)
Target: Binary classification (malignant/benign)
Feature names: ['mean radius' 'mean texture' 'mean perimeter']...
Train: 455 samples, Test: 114 samples


In [3]:
# Build basic pipeline: StandardScaler → LogisticRegression
# This ensures scaler is fit ONLY on training data, preventing leakage
basic_pipe = Pipeline([
    ('scaler', StandardScaler()),           # Step 1: Standardize features
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))  # Step 2: Classify
])

# The pipeline behaves like a single estimator
print("Pipeline steps:")
for name, step in basic_pipe.named_steps.items():
    print(f"  {name}: {step.__class__.__name__}")

# Fit on training data
basic_pipe.fit(X_train, y_train)

# Predict on test data (scaler uses training params automatically)
y_pred = basic_pipe.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)

print(f"\nTest accuracy: {test_acc:.4f}")

# Access fitted scaler parameters (learned from training data only)
scaler_mean = basic_pipe.named_steps['scaler'].mean_
scaler_scale = basic_pipe.named_steps['scaler'].scale_
print(f"Scaler mean (first 3 features): {scaler_mean[:3]}")
print(f"Scaler scale (first 3 features): {scaler_scale[:3]}")

Pipeline steps:
  scaler: StandardScaler
  classifier: LogisticRegression

Test accuracy: 0.9825
Scaler mean (first 3 features): [14.06721319 19.24736264 91.55740659]
Scaler scale (first 3 features): [ 3.49553212  4.40044714 24.12267871]


In [4]:
# Compare: Manual preprocessing (DANGEROUS - shows leakage risk)
# This is what NOT to do - fitting scaler on full data
scaler_leaky = StandardScaler()
X_scaled_leaky = scaler_leaky.fit_transform(X)  # FIT ON FULL DATA - LEAKAGE!
X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_scaled_leaky, y, test_size=0.2, random_state=42, stratify=y
)

clf_leaky = LogisticRegression(max_iter=1000, random_state=42)
clf_leaky.fit(X_train_leaky, y_train_leaky)
acc_leaky = accuracy_score(y_test_leaky, clf_leaky.predict(X_test_leaky))

# Safe cross-validation with pipeline
cv_scores = cross_val_score(basic_pipe, X, y, cv=5, scoring='accuracy')

print(f"Leaky manual preprocessing accuracy: {acc_leaky:.4f}")
print(f"Safe pipeline CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")
print(f"\nNote: Leaky version may show inflated performance due to test data influencing scaling")
print(f"Pipeline ensures scaler fits only on training folds during CV")

Leaky manual preprocessing accuracy: 0.9825
Safe pipeline CV accuracy: 0.9807 (+/- 0.0131)

Note: Leaky version may show inflated performance due to test data influencing scaling
Pipeline ensures scaler fits only on training folds during CV


## 🏗️ 2. ColumnTransformer – Mixed Data Types

Real datasets have heterogeneous columns: numeric features need scaling/imputation, categorical features need encoding. ColumnTransformer applies different transformers to different columns, then concatenates results.

In [5]:
# Load California housing and add synthetic categorical feature
housing = fetch_california_housing()
X_housing = pd.DataFrame(housing.data, columns=housing.feature_names)
y_housing = housing.target

# Add synthetic categorical feature (like ocean_proximity)
# Bin median income into categories
X_housing['income_category'] = pd.cut(
    X_housing['MedInc'], 
    bins=[0, 2, 4, 6, np.inf],
    labels=['low', 'medium', 'high', 'very_high']
)

# Introduce some missing values for demonstration
rng = np.random.RandomState(42)
missing_idx = rng.choice(X_housing.index, size=50, replace=False)
X_housing.loc[missing_idx[:25], 'AveRooms'] = np.nan
X_housing.loc[missing_idx[25:], 'AveBedrms'] = np.nan

print("California Housing with categorical feature:")
print(X_housing.head())
print(f"\nMissing values: \n{X_housing.isnull().sum()}")

# Split
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_housing, y_housing, test_size=0.2, random_state=42
)

# Identify column types
numeric_features = X_housing.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = ['income_category']
numeric_features = [f for f in numeric_features if f != 'income_category']  # Remove if added

print(f"\nNumeric features ({len(numeric_features)}): {numeric_features[:3]}...")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

California Housing with categorical feature:
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude income_category  
0    -122.23       very_high  
1    -122.22       very_high  
2    -122.24       very_high  
3    -122.25            high  
4    -122.25          medium  

Missing values: 
MedInc              0
HouseAge            0
AveRooms           25
AveBedrms          25
Population          0
AveOccup            0
Latitude            0
Longitude           0
income_category     0
dtype: int64

Numeric features (8): ['MedInc', 'HouseAge', 'AveRooms']...
Categorical features (

In [6]:
# Build preprocessing pipelines for each column type
# Numeric pipeline: impute missing values, then scale
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # Fill NaNs with median
    ('scaler', StandardScaler())                     # Standardize
])

# Categorical pipeline: encode as one-hot
categorical_transformer = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    # handle_unknown='ignore' prevents crashes on unseen categories in production
])

# Combine with ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
], remainder='drop')  # 'drop' removes unspecified columns; 'passthrough' keeps them

# Full pipeline: preprocessing + model
full_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

print("Full pipeline structure:")
print(full_pipe)

# Fit and evaluate
full_pipe.fit(X_train_h, y_train_h)
y_pred_h = full_pipe.predict(X_test_h)
rmse = np.sqrt(mean_squared_error(y_test_h, y_pred_h))

print(f"\nTest RMSE: {rmse:.4f}")
print(f"Preprocessing handled missing values and categorical encoding automatically")

Full pipeline structure:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['MedInc', 'HouseAge',
                                                   'AveRooms', 'AveBedrms',
                                                   'Population', 'AveOccup',
                                                   'Latitude', 'Longitude']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='igno

In [7]:
# Inspect the preprocessor after fitting
# Get feature names after transformation
preprocessor_fitted = full_pipe.named_steps['preprocessor']

# Numeric feature names (unchanged)
num_features = numeric_features

# Categorical feature names (expanded by one-hot encoding)
cat_encoder = preprocessor_fitted.named_transformers_['cat'].named_steps['encoder']
cat_features = cat_encoder.get_feature_names_out(categorical_features)

# Combined feature names
all_features = list(num_features) + list(cat_features)
print(f"Total features after preprocessing: {len(all_features)}")
print(f"Feature names: {all_features[:5]}...{all_features[-3:]}")

# Verify no leakage: check that imputer statistics come from training data only
imputer_stats = preprocessor_fitted.named_transformers_['num'].named_steps['imputer'].statistics_
print(f"\nImputer statistics (median values from training data):")
for feat, stat in zip(numeric_features[:3], imputer_stats[:3]):
    print(f"  {feat}: {stat:.2f}")

Total features after preprocessing: 12
Feature names: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population']...['income_category_low', 'income_category_medium', 'income_category_very_high']

Imputer statistics (median values from training data):
  MedInc: 3.55
  HouseAge: 29.00
  AveRooms: 5.24


## 🔧 3. Custom Transformers – Safe Feature Engineering

Domain-specific features often require custom transformations. By inheriting from `BaseEstimator` and `TransformerMixin`, we create transformers that integrate seamlessly with pipelines and follow the same fit/transform contract.

In [8]:
# Custom transformer: Add engineered ratio features
class RatioAdder(BaseEstimator, TransformerMixin):
    """
    Custom transformer that adds ratio features to housing data.
    Inherits from BaseEstimator and TransformerMixin for pipeline compatibility.
    """
    def __init__(self, add_bedrooms_per_room=True):
        # Hyperparameter: control which features to add
        self.add_bedrooms_per_room = add_bedrooms_per_room
    
    def fit(self, X, y=None):
        """
        Fit method - nothing to learn for this stateless transformer.
        Returns self for pipeline compatibility.
        """
        return self  # Nothing to fit
    
    def transform(self, X):
        """
        Transform method - add new ratio features.
        X can be numpy array or DataFrame.
        """
        # Convert to DataFrame if needed (to access column names)
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=['MedInc', 'HouseAge', 'AveRooms', 
                                        'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude'])
        
        X = X.copy()  # Don't modify original
        
        # Add rooms per household (avg rooms / avg occupancy proxy)
        X['rooms_per_household'] = X['AveRooms'] / X['AveOccup']
        
        # Add population per household
        X['pop_per_household'] = X['Population'] / X['AveOccup']
        
        if self.add_bedrooms_per_room:
            # Add bedrooms per room ratio
            X['bedrooms_per_room'] = X['AveBedrms'] / X['AveRooms']
        
        return X.values if isinstance(X, pd.DataFrame) else X

# Test custom transformer
ratio_adder = RatioAdder(add_bedrooms_per_room=True)
X_sample = X_train_h[numeric_features].iloc[:5]
X_transformed = ratio_adder.fit_transform(X_sample)

print(f"Original shape: {X_sample.shape}")
print(f"Transformed shape: {X_transformed.shape}")
print(f"Added {X_transformed.shape[1] - X_sample.shape[1]} new ratio features")

# Show feature names (we'd track these separately in practice)
print("\nFirst sample with new features:")
print(f"  rooms_per_household: {X_transformed[0, -3]:.2f}")
print(f"  pop_per_household: {X_transformed[0, -2]:.2f}")
print(f"  bedrooms_per_room: {X_transformed[0, -1]:.3f}")

Original shape: (5, 8)
Transformed shape: (5, 11)
Added 3 new ratio features

First sample with new features:
  rooms_per_household: 1.36
  pop_per_household: 623.00
  bedrooms_per_room: 0.201


In [9]:
# Integrate custom transformer into full pipeline
# New numeric pipeline with custom transformer
numeric_transformer_custom = Pipeline([
    ('ratio_adder', RatioAdder(add_bedrooms_per_room=True)),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Updated preprocessor
preprocessor_custom = ColumnTransformer([
    ('num', numeric_transformer_custom, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Full pipeline with custom features
custom_pipe = Pipeline([
    ('preprocessor', preprocessor_custom),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Train and evaluate
custom_pipe.fit(X_train_h, y_train_h)
y_pred_custom = custom_pipe.predict(X_test_h)
rmse_custom = np.sqrt(mean_squared_error(y_test_h, y_pred_custom))

print(f"Custom pipeline RMSE: {rmse_custom:.4f}")
print(f"Previous RMSE (no custom features): {rmse:.4f}")
print(f"Improvement: {rmse - rmse_custom:.4f}")

# Key point: Custom transformer fits on train, transforms test safely
# No data leakage because ratio_adder is stateless (no fit logic)
# If it learned statistics, they would be computed from train only

Custom pipeline RMSE: 0.5059
Previous RMSE (no custom features): 0.5061
Improvement: 0.0002


## 🔀 4. FeatureUnion – Combining Multiple Feature Sets

FeatureUnion applies multiple transformers to the same input data and concatenates results horizontally. This enables combining raw features with dimensionality reduction (PCA) and polynomial expansions in one pipeline.

In [10]:
# Create FeatureUnion: combine PCA, polynomial features, and raw features
feature_union = FeatureUnion([
    ('pca', PCA(n_components=5)),           # 5 principal components
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),  # Quadratic interactions
    ('raw', 'passthrough')                     # Original features unchanged
])

# Test on scaled numeric data
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train_h[numeric_features])
X_test_imputed = imputer.transform(X_test_h[numeric_features])
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# Fit union and transform
X_train_union = feature_union.fit_transform(X_train_scaled)
X_test_union = feature_union.transform(X_test_scaled)

print("FeatureUnion results:")
print(f"  Input shape: {X_train_scaled.shape}")
print(f"  PCA output: 5 features")
print(f"  Poly output: {PolynomialFeatures(2).fit(X_train_scaled).n_output_features_} features")
print(f"  Raw output: {X_train_scaled.shape[1]} features")
print(f"  Union output shape: {X_train_union.shape}")
print(f"  Total new features created: {X_train_union.shape[1] - X_train_scaled.shape[1]}")

# Full pipeline with ColumnTransformer + FeatureUnion
# For simplicity, apply union to numeric features only
numeric_union_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('feature_union', feature_union)
])

final_preprocessor = ColumnTransformer([
    ('num', numeric_union_pipe, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

union_pipe = Pipeline([
    ('preprocessor', final_preprocessor),
    ('regressor', Ridge(alpha=1.0))
    # Using Ridge because high-dimensional poly features can cause multicollinearity
])

# Train
union_pipe.fit(X_train_h, y_train_h)
y_pred_union = union_pipe.predict(X_test_h)
rmse_union = np.sqrt(mean_squared_error(y_test_h, y_pred_union))

print(f"\nFeatureUnion + Ridge RMSE: {rmse_union:.4f}")
print(f"RandomForest (no union) RMSE: {rmse:.4f}")
print("Note: Feature engineering helps linear models more than tree-based models")

FeatureUnion results:
  Input shape: (16512, 8)
  PCA output: 5 features
  Poly output: 45 features
  Raw output: 8 features
  Union output shape: (16512, 57)
  Total new features created: 49

FeatureUnion + Ridge RMSE: 0.6893
RandomForest (no union) RMSE: 0.5061
Note: Feature engineering helps linear models more than tree-based models


## 💾 5. Caching & Memory – Speed Up Repeated CV

Expensive preprocessing steps (text vectorization, heavy imputation) shouldn't repeat on every CV fold. Pipeline's `memory` parameter caches fitted transformers to disk, dramatically speeding up grid search.

In [11]:
# Create a slow transformer to demonstrate caching benefit
class SlowTransformer(BaseEstimator, TransformerMixin):
    """Artificially slow transformer to show caching effect."""
    def __init__(self, sleep_time=0.1):
        self.sleep_time = sleep_time
    
    def fit(self, X, y=None):
        time.sleep(self.sleep_time)  # Simulate expensive computation
        return self
    
    def transform(self, X):
        time.sleep(self.sleep_time / 2)
        return X

# Setup cache directory
from tempfile import mkdtemp
cachedir = mkdtemp(prefix='sklearn_cache_')
memory = Memory(location=cachedir, verbose=0)

# Pipeline without caching
slow_pipe_no_cache = Pipeline([
    ('slow', SlowTransformer(sleep_time=0.5)),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Pipeline with caching
slow_pipe_cache = Pipeline([
    ('slow', SlowTransformer(sleep_time=0.5)),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000))
], memory=memory)  # Enable caching!

# Use small dataset for demo
X_small, y_small = X[:100], y[:100]

print("Running 3-fold CV without cache...")
start = time.time()
scores_no_cache = cross_val_score(slow_pipe_no_cache, X_small, y_small, cv=3)
time_no_cache = time.time() - start

print(f"  Time: {time_no_cache:.2f}s, Accuracy: {scores_no_cache.mean():.4f}")

print("\nRunning 3-fold CV with cache (first run - populates cache)...")
start = time.time()
scores_cache_1 = cross_val_score(slow_pipe_cache, X_small, y_small, cv=3)
time_cache_1 = time.time() - start
print(f"  Time: {time_cache_1:.2f}s, Accuracy: {scores_cache_1.mean():.4f}")

print("\nRunning 3-fold CV with cache (second run - uses cache)...")
start = time.time()
scores_cache_2 = cross_val_score(slow_pipe_cache, X_small, y_small, cv=3)
time_cache_2 = time.time() - start
print(f"  Time: {time_cache_2:.2f}s, Accuracy: {scores_cache_2.mean():.4f}")

print(f"\nSpeedup: {time_no_cache / time_cache_2:.1f}x faster with caching")

Running 3-fold CV without cache...
  Time: 3.05s, Accuracy: 0.9397

Running 3-fold CV with cache (first run - populates cache)...
  Time: 3.19s, Accuracy: 0.9397

Running 3-fold CV with cache (second run - uses cache)...
  Time: 1.05s, Accuracy: 0.9397

Speedup: 2.9x faster with caching


In [12]:
# Real example: GridSearchCV with caching on housing data
# Cache the expensive preprocessing step
preprocessor_cache = ColumnTransformer([
    ('num', numeric_transformer_custom, numeric_features),
    ('cat', categorical_transformer, categorical_features)
], remainder='drop')

# Pipeline with caching for GridSearch
search_pipe = Pipeline([
    ('preprocessor', preprocessor_cache),
    ('regressor', RandomForestRegressor(random_state=42))
], memory=memory)

# Parameter grid
param_grid = {
    'regressor__n_estimators': [50, 100],
    'regressor__max_depth': [10, 20, None]
}

print("Running GridSearchCV with caching...")
print("Preprocessing will be cached across all parameter combinations!")

grid_search = GridSearchCV(
    search_pipe, param_grid,
    cv=3, scoring='neg_mean_squared_error',
    n_jobs=1, verbose=1
)

start = time.time()
grid_search.fit(X_train_h, y_train_h)
search_time = time.time() - start

print(f"\nGrid search completed in {search_time:.1f}s")
print(f"Best params: {grid_search.best_params_}")
print(f"Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}")

# Clear cache when done
memory.clear(warn=False)
print("Cache cleared")

Running GridSearchCV with caching...
Preprocessing will be cached across all parameter combinations!
Fitting 3 folds for each of 6 candidates, totalling 18 fits

Grid search completed in 443.9s
Best params: {'regressor__max_depth': None, 'regressor__n_estimators': 100}
Best CV RMSE: 0.5193
Cache cleared


## 📦 6. Serialization & Deployment Readiness

Production deployment requires saving the complete workflow. `joblib` serializes the entire pipeline (preprocessing + model) so you can load and predict on new data without any code duplication.

In [13]:
# Save the complete pipeline to disk
model_filename = 'production_pipeline.joblib'

# Save fitted pipeline (preprocessor + trained model)
joblib.dump(custom_pipe, model_filename)
print(f"Pipeline saved to: {model_filename}")
print(f"File size: {np.round(os.path.getsize(model_filename) / 1024, 2)} KB")

# In production environment, load and predict
# No preprocessing code needed - it's all in the pipeline!
loaded_pipe = joblib.load(model_filename)
print(f"\nPipeline loaded successfully")

# Generate predictions on new data
# In production, X_new would come from API request, database, etc.
X_new = X_test_h.iloc[:5]  # Simulate new data
predictions = loaded_pipe.predict(X_new)

print(f"Predictions for 5 new samples: {predictions}")
print(f"Pipeline automatically applied all preprocessing steps:")
print(f"  1. Custom ratio features")
print(f"  2. Missing value imputation")
print(f"  3. Categorical encoding")
print(f"  4. Feature scaling")
print(f"  5. Random Forest prediction")

# Verify predictions match original pipeline
original_preds = custom_pipe.predict(X_new)
assert np.allclose(predictions, original_preds), "Mismatch in predictions!"
print("\n✓ Loaded pipeline produces identical predictions to original")

Pipeline saved to: production_pipeline.joblib
File size: 141225.85 KB

Pipeline loaded successfully
Predictions for 5 new samples: [0.66571   0.7462    4.4130822 2.59055   2.23505  ]
Pipeline automatically applied all preprocessing steps:
  1. Custom ratio features
  2. Missing value imputation
  3. Categorical encoding
  4. Feature scaling
  5. Random Forest prediction

✓ Loaded pipeline produces identical predictions to original


In [14]:
# Pipeline inspection and hyperparameter access
# Get all parameters (useful for logging/tracking)
all_params = loaded_pipe.get_params()
print("Pipeline parameters (sample):")
for key in list(all_params.keys())[:10]:
    print(f"  {key}: {all_params[key]}")

# Modify parameters dynamically (for A/B testing, gradual rollouts)
loaded_pipe.set_params(regressor__n_estimators=200)
print(f"\nUpdated n_estimators to 200")
print(f"New regressor params: {loaded_pipe.named_steps['regressor'].get_params()['n_estimators']}")

# Feature importance (if available)
regressor = loaded_pipe.named_steps['regressor']
if hasattr(regressor, 'feature_importances_'):
    importances = regressor.feature_importances_
    print(f"\nTop 3 important features (indices): {importances.argsort()[-3:][::-1]}")

# Clean up
import os
os.remove(model_filename)
print(f"\nCleaned up {model_filename}")

Pipeline parameters (sample):
  memory: None
  steps: [('preprocessor', ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('ratio_adder', RatioAdder()),
                                                 ('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
                                  'Population', 'AveOccup', 'Latitude',
                                  'Longitude']),
                                ('cat',
                                 Pipeline(steps=[('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['income_category'])])), ('regressor', RandomForestRegressor(random

## 🎯 7. Full Realistic Pipeline – End-to-End Example

Combine everything: custom transformers, ColumnTransformer, FeatureUnion, feature selection, and hyperparameter tuning in one robust, reproducible workflow.

In [15]:
# Complete end-to-end pipeline with all techniques
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import GradientBoostingRegressor

# 1. Custom transformer (domain-specific features)
class HousingFeatureEngineer(BaseEstimator, TransformerMixin):
    """Advanced feature engineering for housing data."""
    def __init__(self, coastal=True):
        self.coastal = coastal

    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = pd.DataFrame(X, columns=numeric_features) if isinstance(X, np.ndarray) else X.copy()
        X['rooms_per_pop'] = X['AveRooms'] / (X['Population'] + 1)  # Avoid div by zero
        X['bedroom_ratio'] = X['AveBedrms'] / (X['AveRooms'] + 0.1)
        if self.coastal:
            X['coastal'] = ((X['Latitude'] < 37) & (X['Longitude'] < -118)).astype(int)
        return X.values if hasattr(X, 'values') else X

# 2. Numeric pipeline with feature engineering
numeric_pipe_final = Pipeline([
    ('engineer', HousingFeatureEngineer()),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 3. Preprocessor
preprocessor_final = ColumnTransformer([
    ('num', numeric_pipe_final, numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
])

# 4. Full pipeline with feature selection
# SelectFromModel uses a base estimator to select important features
selector = SelectFromModel(
    estimator=RandomForestRegressor(n_estimators=50, random_state=42),
    max_features=15  # Keep top 15 features
)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_final),
    ('feature_selection', selector),
    ('regressor', GradientBoostingRegressor(random_state=42))
])

print("Final pipeline structure:")
for name, step in final_pipeline.named_steps.items():
    print(f"  {name}: {step.__class__.__name__}")

# Train on full training data
final_pipeline.fit(X_train_h, y_train_h)

# Evaluate
y_pred_final = final_pipeline.predict(X_test_h)
rmse_final = np.sqrt(mean_squared_error(y_test_h, y_pred_final))

print(f"\nFinal pipeline RMSE: {rmse_final:.4f}")
print(f"Improvement over baseline: {rmse - rmse_final:.4f} ({(rmse - rmse_final)/rmse*100:.1f}%)")

Final pipeline structure:
  preprocessor: ColumnTransformer
  feature_selection: SelectFromModel
  regressor: GradientBoostingRegressor

Final pipeline RMSE: 0.5464
Improvement over baseline: -0.0404 (-8.0%)


In [16]:
# Grid search on the final pipeline
param_grid_final = {
    'preprocessor__num__engineer__coastal': [True, False],  # Toggle custom feature
    'feature_selection__max_features': [10, 15, 20],
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [3, 5, 7]
}

print("Running final grid search (this may take a minute)...")
grid_final = GridSearchCV(
    final_pipeline,
    param_grid_final,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=1,
    verbose=1
)

grid_final.fit(X_train_h, y_train_h)

# Best results
best_rmse = np.sqrt(-grid_final.best_score_)
print(f"\nBest CV RMSE: {best_rmse:.4f}")
print(f"Best parameters: {grid_final.best_params_}")

# Test set performance
y_pred_best = grid_final.predict(X_test_h)
test_rmse_best = np.sqrt(mean_squared_error(y_test_h, y_pred_best))
print(f"Test RMSE with best params: {test_rmse_best:.4f}")

# Summary
print("\n" + "="*60)
print("PIPELINE EVOLUTION SUMMARY")
print("="*60)
results_summary = [
    ("Baseline (no pipeline)", rmse),
    ("Basic ColumnTransformer", rmse),
    ("Custom Features", rmse_custom),
    ("FeatureUnion", rmse_union),
    ("Full Pipeline + Selection", rmse_final),
    ("Tuned Pipeline (GridSearch)", test_rmse_best)
]
for name, score in results_summary:
    print(f"{name:30s}: {score:.4f}")

Running final grid search (this may take a minute)...
Fitting 3 folds for each of 36 candidates, totalling 108 fits

Best CV RMSE: 0.4891
Best parameters: {'feature_selection__max_features': 10, 'preprocessor__num__engineer__coastal': True, 'regressor__max_depth': 7, 'regressor__n_estimators': 200}
Test RMSE with best params: 0.4759

PIPELINE EVOLUTION SUMMARY
Baseline (no pipeline)        : 0.5061
Basic ColumnTransformer       : 0.5061
Custom Features               : 0.5059
FeatureUnion                  : 0.6893
Full Pipeline + Selection     : 0.5464
Tuned Pipeline (GridSearch)   : 0.4759


## ⚠️ Common Pitfalls & Pro Tips

- **Fit scaler on full data → leakage**: Always use Pipeline to ensure transformers fit only on training data during CV
- **Forgetting passthrough in ColumnTransformer**: Use `remainder='passthrough'` to keep unspecified columns, or explicitly drop them
- **Custom transformer not implementing fit/transform correctly**: Must inherit from `BaseEstimator` and `TransformerMixin`, return `self` from fit
- **Caching stale data → wrong results**: Clear cache (`memory.clear()`) when data or transformer logic changes
- **Not using memory= in long CV**: GridSearchCV with heavy preprocessing benefits enormously from caching; always enable for production tuning
- **Joblib with large models → compression**: Use `joblib.dump(pipe, 'model.pkl', compress=3)` for large models to reduce disk usage
- **Not handling unknown categories in production**: Set `handle_unknown='ignore'` in OneHotEncoder to prevent crashes on unseen categories
- **Pipeline modification after fitting**: Changing fitted pipeline parameters without refitting causes inconsistent state; clone before modification
- **Feature names lost in numpy arrays**: Track feature names separately or use `get_feature_names_out()` from sklearn 1.0+ for interpretability
- **Inconsistent dtypes between train and test**: Ensure categorical encodings see same categories; use `categories='auto'` and consistent data cleaning
- **Not validating pipeline steps**: Use `set_params` carefully; invalid parameter names fail silently in some sklearn versions
- **Forgetting to set random_state**: Non-deterministic algorithms (RF, GBM) produce different results across runs; set seeds for reproducibility

## 📝 Exercises

Practice building production-ready pipelines:

**Easy:** Build a Pipeline for the Iris dataset: StandardScaler → SVC. Use `cross_val_score` to evaluate and compare with manual preprocessing.

**Medium:** Add a custom log-transformer for skewed features in California Housing. The transformer should apply `np.log1p` to features with skew > 1.0 (learned during fit). Integrate into ColumnTransformer.

**Medium:** Use FeatureUnion to combine PCA (n=3) + raw features on the Breast Cancer dataset. Compare pipeline performance to using raw features only with LogisticRegression.

**Hard:** Create a pipeline with outlier removal (custom transformer using IQR method) → RFE (Recursive Feature Elimination) → final RandomForest model. Ensure outlier removal statistics are learned from training data only.

**Bonus:** Serialize your best pipeline with joblib, then write a function `predict_new_data(pipeline_path, X_new)` that loads the pipeline and returns predictions. Simulate production inference with new data samples.

<details>
<summary><b>Exercise Solutions (Click to Expand)</b></summary>

### Easy Solution Outline (Iris Pipeline)
```python
from sklearn.svm import SVC
iris = load_iris()
pipe = Pipeline([('scaler', StandardScaler()), ('svc', SVC())])
scores = cross_val_score(pipe, iris.data, iris.target, cv=5)
print(f"CV Accuracy: {scores.mean():.3f}")
```

### Medium Solution Outline (Log Transformer)
```python
class LogTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # Calculate skewness for each feature
        self.skew_ = pd.DataFrame(X).skew().values
        self.transform_cols_ = self.skew_ > 1.0
        return self
    
    def transform(self, X):
        X = X.copy()
        for i, should_transform in enumerate(self.transform_cols_):
            if should_transform:
                X[:, i] = np.log1p(np.clip(X[:, i], 0, None))
        return X
```

### Hard Solution Outline (Outlier Removal + RFE)
```python
class OutlierRemover(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # Learn IQR bounds from training data
        Q1 = np.percentile(X, 25, axis=0)
        Q3 = np.percentile(X, 75, axis=0)
        self.lower_ = Q1 - 1.5 * (Q3 - Q1)
        self.upper_ = Q3 + 1.5 * (Q3 - Q1)
        return self
    
    def transform(self, X):
        # Clip to bounds (don't remove rows to maintain array shape)
        return np.clip(X, self.lower_, self.upper_)

# Pipeline: OutlierRemover → RFE → RandomForest
pipe = Pipeline([
    ('outlier', OutlierRemover()),
    ('rfe', RFE(RandomForestClassifier(), n_features_to_select=10)),
    ('rf', RandomForestClassifier())
])
```

### Bonus Solution Outline (Production Function)
```python
def predict_new_data(pipeline_path, X_new):
    pipe = joblib.load(pipeline_path)
    # Ensure X_new is DataFrame with same columns as training
    if isinstance(X_new, dict):
        X_new = pd.DataFrame([X_new])
    return pipe.predict(X_new)

# Usage
new_sample = {'MedInc': 8.3, 'HouseAge': 41, 'AveRooms': 6.9, ...}
pred = predict_new_data('production_pipeline.joblib', new_sample)
```

</details>

## ✅ Summary – What You Learned Today

- **Pipeline fundamentals**: Chain transformers and estimators into single reproducible workflows that prevent data leakage
- **ColumnTransformer**: Handle heterogeneous data (numeric + categorical) with different preprocessing in one unified pipeline
- **Custom transformers**: Implement domain-specific feature engineering by inheriting from `BaseEstimator` and `TransformerMixin`
- **FeatureUnion**: Combine multiple feature extraction strategies (PCA, polynomial, raw) horizontally into expanded feature sets
- **Caching with memory**: Speed up GridSearchCV by caching expensive preprocessing steps across parameter combinations
- **Serialization with joblib**: Save complete workflows (preprocessing + model) for production deployment and reproducibility
- **Safe cross-validation**: Pipelines ensure transformers fit only on training folds, preventing subtle data leakage
- **Production readiness**: Load saved pipelines and generate predictions on new data without reimplementing preprocessing
- **Debugging techniques**: Access `named_steps`, inspect fitted attributes, and use `get_params()`/`set_params()` for introspection
- **End-to-end workflow**: Combine all techniques into robust, tunable, deployable machine learning systems
